In [ ]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código



/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


In [ ]:
!apt-get install -y xvfb libgl1-mesa-glx libgl1-mesa-dri

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1-mesa-dri is already the newest version (23.2.1-1ubuntu3.1~22.04.3).
libgl1-mesa-dri set to manually installed.
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
!pip install pyglet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.5 MB/s eta 0:00:00


In [ ]:
!pip install rware

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.8 MB/s eta 0:00:00
  Attempting uninstall: pyglet
    Found existing installation: pyglet 2.1.14
    Uninstalling pyglet-2.1.14:
      Successfully uninstalled pyglet-2.1.14


In [ ]:
%ls

'Ambiente e Execução IDQN - Versão 1.1.0.py'  'Executa - Experimento.ipynb'
'Ambiente e Execução IDQN - Versão 1.2.0.py'  'Executa IDQN - Versão 7.0.0.py'
'Armazem - Experimento.ipynb'


In [ ]:
# PASSO 1: Instalar o pacote rware no Colab

# PASSO 2: Importar bibliotecas
import gymnasium as gym
import rware  # ESSE IMPORT É OBRIGATÓRIO
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import random
import imageio
from tqdm import tqdm
import os
from datetime import datetime
from google.colab import files
import pandas as pd

# Configurações - CORRIGIDO PARA 6 AGENTES
ENV_NAME = "rware:rware-medium-6ag-hard-v2"  # Ambiente com 6 agentes
NUM_AGENTS = 6  # ← CORRIGIDO: 6 agentes (padrão do ambiente)
EPISODES = 3000
MAX_STEPS = 3000
LEARNING_RATE = 0.0005
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 30000  # Aumentado para mais agentes
BATCH_SIZE = 128
BUFFER_SIZE = 50000
TARGET_UPDATE = 500
HIDDEN_DIM = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Criar diretório para resultados
RESULTS_DIR = "resultados_treinamento"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Usando dispositivo: {DEVICE}")
print(f"Número de agentes: {NUM_AGENTS}")
print(f"Resultados serão salvos em: {RESULTS_DIR}")

# Rede Neural para DQN
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, output_dim)
        )

    def forward(self, x):
        return self.network(x)

# Experience Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))

    def __len__(self):
        return len(self.buffer)

# DQN Agent
class DQNAgent:
    def __init__(self, state_dim, action_dim, agent_id):
        self.agent_id = agent_id
        self.policy_net = DQN(state_dim, action_dim).to(DEVICE)
        self.target_net = DQN(state_dim, action_dim).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = ReplayBuffer(BUFFER_SIZE)
        self.steps_done = 0
        self.action_dim = action_dim

    def select_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()

    def optimize(self):
        if len(self.memory) < BATCH_SIZE:
            return 0

        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)

        states = torch.FloatTensor(states).to(DEVICE)
        actions = torch.LongTensor(actions).to(DEVICE)
        rewards = torch.FloatTensor(rewards).to(DEVICE)
        next_states = torch.FloatTensor(next_states).to(DEVICE)
        dones = torch.FloatTensor(dones).to(DEVICE)

        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1))
        next_q = self.target_net(next_states).max(1)[0].detach()
        target_q = rewards + GAMMA * next_q * (1 - dones)

        loss = nn.MSELoss()(current_q.squeeze(), target_q)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

# Função para processar observação
def process_observation(obs):
    """Converte observação multiagente em vetor flat para cada agente"""
    processed = []
    for agent_obs in obs:
        if isinstance(agent_obs, dict):
            flat = np.concatenate([v.flatten() if hasattr(v, 'flatten') else np.array([v])
                                   for v in agent_obs.values()])
        elif isinstance(agent_obs, np.ndarray):
            flat = agent_obs.flatten()
        else:
            flat = np.array(agent_obs).flatten()

        # Limitar tamanho para consistência
        processed.append(flat[:128])
    return processed

# Função para salvar todos os gráficos
def save_all_plots(episode_returns, epsilon_history, step_counts,
                   episode_deliveries, success_rates, agent_returns,
                   agent_deliveries, episode_losses=None):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # 1. GRÁFICO: Reward Total por Episódio
    plt.figure(figsize=(12, 6))
    plt.plot(episode_returns, alpha=0.5, linewidth=0.8, label='Reward por Episódio', color='blue')

    window = min(50, len(episode_returns) // 10)
    if len(episode_returns) > window:
        moving_avg = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(episode_returns)), moving_avg,
                'r-', linewidth=2, label=f'Média móvel ({window} episódios)')

    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Reward Total', fontsize=12)
    plt.title('Reward Total por Episódio', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    reward_file = os.path.join(RESULTS_DIR, f'reward_total_{timestamp}.png')
    plt.savefig(reward_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 1 salvo: {reward_file}")

    # 2. GRÁFICO: Decaimento do Epsilon
    plt.figure(figsize=(12, 6))
    plt.plot(epsilon_history, color='green', linewidth=1.5)
    plt.xlabel('Passos de Treinamento', fontsize=12)
    plt.ylabel('ε (Epsilon)', fontsize=12)
    plt.title('Decaimento da Taxa de Exploração (ε-greedy)', fontsize=14, fontweight='bold')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    epsilon_file = os.path.join(RESULTS_DIR, f'exploracao_epsilon_{timestamp}.png')
    plt.savefig(epsilon_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 2 salvo: {epsilon_file}")

    # 3. GRÁFICO: Steps por Episódio
    plt.figure(figsize=(12, 6))
    plt.plot(step_counts, color='orange', linewidth=1)
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Número de Steps', fontsize=12)
    plt.title('Steps por Episódio', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)

    if len(step_counts) > window:
        moving_avg_steps = np.convolve(step_counts, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(step_counts)), moving_avg_steps,
                'r-', linewidth=2, label=f'Média móvel ({window} episódios)')
        plt.legend()

    plt.tight_layout()
    steps_file = os.path.join(RESULTS_DIR, f'steps_episodio_{timestamp}.png')
    plt.savefig(steps_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 3 salvo: {steps_file}")

    # 4. GRÁFICO: Número de Entregas
    plt.figure(figsize=(12, 6))
    plt.plot(episode_deliveries, alpha=0.5, linewidth=0.8, label='Entregas por Episódio', color='purple')

    if len(episode_deliveries) > window:
        moving_avg_del = np.convolve(episode_deliveries, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(episode_deliveries)), moving_avg_del,
                'r-', linewidth=2, label=f'Média móvel ({window} episódios)')

    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Número de Entregas', fontsize=12)
    plt.title('Entregas por Episódio', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    deliveries_file = os.path.join(RESULTS_DIR, f'entregas_episodio_{timestamp}.png')
    plt.savefig(deliveries_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 4 salvo: {deliveries_file}")

    # 5. GRÁFICO: Taxa de Sucesso
    plt.figure(figsize=(12, 6))
    plt.plot(success_rates, color='teal', linewidth=1.5)
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Taxa de Sucesso', fontsize=12)
    plt.title('Taxa de Sucesso (Episódios com pelo menos 1 entrega)', fontsize=14, fontweight='bold')
    plt.ylim([0, 1])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    success_file = os.path.join(RESULTS_DIR, f'taxa_sucesso_{timestamp}.png')
    plt.savefig(success_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 5 salvo: {success_file}")

    # 6. GRÁFICO: Recompensa por Robô
    plt.figure(figsize=(12, 6))
    agent_returns_array = np.array(list(agent_returns.values()))

    if len(agent_returns_array.shape) > 1:
        agent_means = agent_returns_array.mean(axis=1)
        agent_stds = agent_returns_array.std(axis=1)
    else:
        agent_means = agent_returns_array
        agent_stds = np.zeros_like(agent_means)

    x_pos = np.arange(NUM_AGENTS)
    bars = plt.bar(x_pos, agent_means, yerr=agent_stds, capsize=5,
                   alpha=0.7, color=plt.cm.tab10(np.linspace(0, 1, NUM_AGENTS)))

    plt.xlabel('Robô', fontsize=12)
    plt.ylabel('Recompensa Média', fontsize=12)
    plt.title('Recompensa Média por Robô', fontsize=14, fontweight='bold')
    plt.xticks(x_pos, [f'Robô {i+1}' for i in range(NUM_AGENTS)])
    plt.grid(True, alpha=0.3, axis='y')

    for bar, value in zip(bars, agent_means):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(agent_stds)/2,
                f'{value:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    agent_reward_file = os.path.join(RESULTS_DIR, f'recompensa_por_robo_{timestamp}.png')
    plt.savefig(agent_reward_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 6 salvo: {agent_reward_file}")

    # 7. GRÁFICO: Entregas por Robô
    plt.figure(figsize=(12, 6))
    agent_del_array = np.array(list(agent_deliveries.values()))

    if len(agent_del_array.shape) > 1:
        del_means = agent_del_array.mean(axis=1)
        del_stds = agent_del_array.std(axis=1)
    else:
        del_means = agent_del_array
        del_stds = np.zeros_like(del_means)

    bars = plt.bar(x_pos, del_means, yerr=del_stds, capsize=5,
                   alpha=0.7, color=plt.cm.tab10(np.linspace(0, 1, NUM_AGENTS)))

    plt.xlabel('Robô', fontsize=12)
    plt.ylabel('Entregas Médias por Episódio', fontsize=12)
    plt.title('Entregas Médias por Robô', fontsize=14, fontweight='bold')
    plt.xticks(x_pos, [f'Robô {i+1}' for i in range(NUM_AGENTS)])
    plt.grid(True, alpha=0.3, axis='y')

    for bar, value in zip(bars, del_means):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(del_stds)/2,
                f'{value:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    agent_deliveries_file = os.path.join(RESULTS_DIR, f'entregas_por_robo_{timestamp}.png')
    plt.savefig(agent_deliveries_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 7 salvo: {agent_deliveries_file}")

    # 8. GRÁFICO COMPARATIVO
    fig, ax1 = plt.subplots(figsize=(12, 6))

    if len(episode_returns) > window:
        smooth_returns = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
        smooth_deliveries = np.convolve(episode_deliveries, np.ones(window)/window, mode='valid')
        x_range = range(window-1, len(episode_returns))
    else:
        smooth_returns = episode_returns
        smooth_deliveries = episode_deliveries
        x_range = range(len(episode_returns))

    color1 = 'blue'
    ax1.set_xlabel('Episódio', fontsize=12)
    ax1.set_ylabel('Recompensa Total', color=color1, fontsize=12)
    ax1.plot(x_range, smooth_returns, color=color1, linewidth=2, label='Recompensa')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    color2 = 'green'
    ax2.set_ylabel('Número de Entregas', color=color2, fontsize=12)
    ax2.plot(x_range, smooth_deliveries, color=color2, linewidth=2, label='Entregas')
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title('Relação entre Recompensa e Entregas (Média Móvel)', fontsize=14, fontweight='bold')
    fig.tight_layout()
    comparative_file = os.path.join(RESULTS_DIR, f'comparativo_recompensa_entregas_{timestamp}.png')
    plt.savefig(comparative_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico 8 salvo: {comparative_file}")

    # Salvar dados em CSV
    df_metrics = pd.DataFrame({
        'episodio': range(1, len(episode_returns) + 1),
        'reward_total': episode_returns,
        'steps': step_counts,
        'entregas': episode_deliveries,
        'taxa_sucesso': success_rates
    })
    csv_file = os.path.join(RESULTS_DIR, f'metricas_treinamento_{timestamp}.csv')
    df_metrics.to_csv(csv_file, index=False)
    print(f"✓ Dados salvos em CSV: {csv_file}")

    # Salvar estatísticas resumidas
    stats = {
        'episodios_treinados': len(episode_returns),
        'reward_total_medio_ultimos_100': np.mean(episode_returns[-100:]),
        'reward_total_std_ultimos_100': np.std(episode_returns[-100:]),
        'entregas_medias_ultimos_100': np.mean(episode_deliveries[-100:]),
        'taxa_sucesso_final': success_rates[-1],
        'melhor_reward': np.max(episode_returns),
        'total_entregas': np.sum(episode_deliveries)
    }

    stats_df = pd.DataFrame([stats])
    stats_file = os.path.join(RESULTS_DIR, f'estatisticas_resumo_{timestamp}.csv')
    stats_df.to_csv(stats_file, index=False)
    print(f"✓ Estatísticas salvas: {stats_file}")

    return reward_file, deliveries_file, success_file

# Função para salvar vídeo
def save_video(env, agents, filename, num_steps=500):
    """Gera vídeo da execução dos agentes"""
    frames = []
    obs, _ = env.reset()
    obs = process_observation(obs)

    for step in range(num_steps):
        try:
            frame = env.render()
            if frame is not None:
                frames.append(frame)

            actions = []
            for i in range(NUM_AGENTS):
                action = agents[i].select_action(obs[i], 0.0)
                actions.append(action)

            obs, rewards, terminated, truncated, _ = env.step(actions)
            obs = process_observation(obs)

            if terminated or truncated:
                break
        except Exception as e:
            print(f"Erro na gravação do vídeo: {e}")
            break

    if frames:
        video_path = os.path.join(RESULTS_DIR, filename)
        with imageio.get_writer(video_path, fps=10, codec='libx264', quality=8) as writer:
            for frame in frames:
                writer.append_data(frame)
        print(f"✓ Vídeo salvo: {video_path}")
        return video_path
    else:
        print("⚠️ Não foi possível capturar frames para o vídeo")
        return None

# Função principal de treinamento
def train():
    print(f"Inicializando ambiente {ENV_NAME}...")

    # Criar ambiente
    env = gym.make(ENV_NAME, render_mode='rgb_array')

    # Obter dimensões
    sample_obs, _ = env.reset()
    sample_obs = process_observation(sample_obs)
    state_dim = len(sample_obs[0])
    action_dim = env.action_space[0].n

    print(f"Dimensão do estado: {state_dim}")
    print(f"Dimensão da ação: {action_dim}")
    print(f"Número de agentes: {NUM_AGENTS}")

    # Criar agentes (um para cada robô)
    agents = [DQNAgent(state_dim, action_dim, i) for i in range(NUM_AGENTS)]

    # Métricas
    episode_returns = []
    epsilon_history = []
    step_counts = []
    episode_deliveries = []
    success_rates = []
    agent_returns_history = {i: [] for i in range(NUM_AGENTS)}
    agent_deliveries_history = {i: [] for i in range(NUM_AGENTS)}

    epsilon = EPSILON_START
    total_steps = 0

    print("\n🚀 Iniciando treinamento...\n")

    for episode in tqdm(range(EPISODES), desc="Treinamento"):
        obs, _ = env.reset()
        obs = process_observation(obs)
        episode_return = 0
        step_count = 0
        deliveries = 0
        agent_ep_returns = [0] * NUM_AGENTS
        agent_ep_deliveries = [0] * NUM_AGENTS

        for step in range(MAX_STEPS):
            # Selecionar ações para todos os agentes
            actions = []
            for i in range(NUM_AGENTS):
                action = agents[i].select_action(obs[i], epsilon)
                actions.append(action)

            # Executar ações no ambiente
            next_obs, rewards, terminated, truncated, info = env.step(actions)
            next_obs = process_observation(next_obs)

            # Armazenar experiências e acumular recompensas
            for i in range(NUM_AGENTS):
                agents[i].memory.push(obs[i], actions[i], rewards[i],
                                      next_obs[i], terminated or truncated)
                episode_return += rewards[i]
                agent_ep_returns[i] += rewards[i]
                if rewards[i] > 0:
                    agent_ep_deliveries[i] += 1
                    deliveries += 1

            # Otimizar agentes
            for i in range(NUM_AGENTS):
                agents[i].optimize()

            step_count += 1
            total_steps += 1

            # Decair epsilon
            if total_steps < EPSILON_DECAY:
                epsilon = EPSILON_START - (EPSILON_START - EPSILON_END) * total_steps / EPSILON_DECAY
            else:
                epsilon = EPSILON_END

            if terminated or truncated:
                break

            obs = next_obs

        # Atualizar target networks periodicamente
        if episode % TARGET_UPDATE == 0 and episode > 0:
            for i in range(NUM_AGENTS):
                agents[i].update_target()

        # Registrar métricas
        episode_returns.append(episode_return)
        epsilon_history.append(epsilon)
        step_counts.append(step_count)
        episode_deliveries.append(deliveries)

        # Calcular taxa de sucesso (episódios com pelo menos 1 entrega)
        success_rate = sum(1 for d in episode_deliveries[-100:] if d > 0) / min(100, len(episode_deliveries))
        success_rates.append(success_rate)

        for i in range(NUM_AGENTS):
            agent_returns_history[i].append(agent_ep_returns[i])
            agent_deliveries_history[i].append(agent_ep_deliveries[i])

        # Logging periódico
        if (episode + 1) % 100 == 0:
            avg_return = np.mean(episode_returns[-100:])
            avg_deliveries = np.mean(episode_deliveries[-100:])
            print(f"\n📈 Episódio {episode+1}: Retorno médio={avg_return:.2f}, "
                  f"Entregas médias={avg_deliveries:.2f}, ε={epsilon:.3f}, "
                  f"Passos={total_steps}, Sucesso={success_rate:.2%}")

    env.close()

    print("\n✅ Treinamento concluído!")
    print("📊 Salvando gráficos...")

    # Salvar todos os gráficos
    reward_file, deliveries_file, success_file = save_all_plots(
        episode_returns, epsilon_history, step_counts,
        episode_deliveries, success_rates, agent_returns_history,
        agent_deliveries_history
    )

    print("\n💾 Salvando modelos...")
    for i, agent in enumerate(agents):
        model_file = os.path.join(RESULTS_DIR, f'dqn_agent_{i}.pth')
        torch.save(agent.policy_net.state_dict(), model_file)
        print(f"  ✓ Modelo Robô {i+1} salvo: {model_file}")

    print("\n🎬 Gerando vídeo da solução...")
    video_env = gym.make(ENV_NAME, render_mode='rgb_array')
    video_file = save_video(video_env, agents, 'solucao_robos.mp4', num_steps=MAX_STEPS)

    # Compactar todos os resultados
    print("\n📦 Compactando resultados...")
    zip_file = f"resultados_treinamento_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
    !zip -r {zip_file} {RESULTS_DIR}/

    # Download no Colab
    print(f"\n📥 Fazendo download do arquivo: {zip_file}")
    files.download(zip_file)

    print("\n✨ TREINAMENTO FINALIZADO COM SUCESSO! ✨")
    print(f"📁 Todos os resultados estão na pasta: {RESULTS_DIR}")
    print(f"📦 Arquivo compactado: {zip_file}")

    return agents, episode_returns, episode_deliveries

# Executar treinamento
if __name__ == "__main__":
    agents, returns, deliveries = train()

Usando dispositivo: cuda
Número de agentes: 6
Resultados serão salvos em: resultados_treinamento
Inicializando ambiente rware:rware-medium-6ag-hard-v2...
Dimensão do estado: 71
Dimensão da ação: 5
Número de agentes: 6

🚀 Iniciando treinamento...



Treinamento:   0%|          | 0/3000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:244: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'list'>
  logger.warn(
Treinamento:   3%|▎         | 100/3000 [16:56<8:31:10, 10.58s/it]


📈 Episódio 100: Retorno médio=0.02, Entregas médias=0.02, ε=0.010, Passos=50000, Sucesso=2.00%


Treinamento:   7%|▋         | 200/3000 [35:11<8:33:37, 11.01s/it]


📈 Episódio 200: Retorno médio=0.00, Entregas médias=0.00, ε=0.010, Passos=100000, Sucesso=0.00%


...


📈 Episódio 3000: Retorno médio=0.00, Entregas médias=0.00, ε=0.010, Passos=1500000, Sucesso=0.00%

✅ Treinamento concluído!
📊 Salvando gráficos...


✓ Gráfico 1 salvo: resultados_treinamento/reward_total_20260604_051758.png
✓ Gráfico 2 salvo: resultados_treinamento/exploracao_epsilon_20260604_051758.png
✓ Gráfico 3 salvo: resultados_treinamento/steps_episodio_20260604_051758.png
✓ Gráfico 4 salvo: resultados_treinamento/entregas_episodio_20260604_051758.png
✓ Gráfico 5 salvo: resultados_treinamento/taxa_sucesso_20260604_051758.png
✓ Gráfico 6 salvo: resultados_treinamento/recompensa_por_robo_20260604_051758.png
✓ Gráfico 7 salvo: resultados_treinamento/entregas_por_robo_20260604_051758.png
✓ Gráfico 8 salvo: resultados_treinamento/comparativo_recompensa_entregas_20260604_051758.png
✓ Dados salvos em CSV: resultados_treinamento/metricas_treinamento_20260604_051758.csv
✓ Estatísticas salvas: resultados_treinamento/estatisticas_resumo_20260604_051758.csv

💾 Salvando modelos...
  ✓ Modelo Robô 1 salvo: resultados_treinamento/dqn_agent_0.pth
  ✓ Modelo Robô 2 salvo: resultados_treinamento/dqn_agent_1.pth
  ✓ Modelo Robô 3 salvo: resulta

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ TREINAMENTO FINALIZADO COM SUCESSO! ✨
📁 Todos os resultados estão na pasta: resultados_treinamento
📦 Arquivo compactado: resultados_treinamento_20260604_051803.zip


In [ ]:
########################################################################
# Primeiro Requerimento
########################################################################
%%time
!python "Executa IDQN - Versão 7.0.0.py"
